# 差動二輪ロボットの運動学と経路追従(演習)

解説資料は `research-handbook/robotics/mobile-robots.md`。
このnotebookは**直接編集せず**、自分の卒研repoにコピーして使うこと(手順は `technical-handbook/colab/use-github-repo.md`)。

「---- ここから課題 ----」の区間を埋めながら上から順に実行する。
解答付き版は `research-handbook/notebooks/diff_drive_solution.ipynb` にあるが、まず自力で取り組むこと。

**このnotebookで学ぶこと**

- 差動二輪の運動学モデルとオイラー積分
- デッドレコニングの誤差蓄積(なぜ自己位置推定が必要か)
- pure pursuit による経路追従と前方注視距離の効果

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

W  = 0.3    # トレッド(左右車輪間距離) [m]
DT = 0.02   # 時間刻み [s]

## 課題1: 運動学モデル

$$\dot{x} = v \cos\theta, \qquad \dot{y} = v \sin\theta, \qquad \dot{\theta} = \omega$$

をオイラー積分で離散化します。角度は $[-\pi, \pi]$ に正規化しておくこと(`mobile-robots.md` 参照)。
「左右同速で直進」「左右差で半径 $v/\omega$ の円」の2つで動作確認します。

In [ ]:
def step(pose, v_l, v_r, w=W, dt=DT):
    """課題1: 差動二輪の運動学(オイラー積分で1ステップ)。
    pose = [x, y, theta]、v_l, v_r は左右車輪の並進速度 [m/s]"""
    x, y, th = pose
    # ---- ここから課題1 ----
    # (1) 車輪速度から並進速度 v と角速度 omega を計算 (mobile-robots.md)
    # (2) x, y, theta をオイラー積分で dt だけ進める
    # (3) theta は [-pi, pi] に正規化する (np.arctan2(np.sin(th), np.cos(th)))


    # ---- ここまで ----
    return np.array([x, y, th])

# 動作確認1: 左右同速 → 直進
pose = np.array([0.0, 0.0, 0.0])
for _ in range(100):
    pose = step(pose, 0.5, 0.5)
print("直進 2s 後:", pose, "(x=1.0, y=0, theta=0 になるはず)")

# 動作確認2: 左右差 → 円運動。v=0.5, omega=1.0 なら半径 v/omega = 0.5 の円
pose = np.array([0.0, 0.0, 0.0])
traj = [pose]
for _ in range(int(2 * np.pi / 1.0 / DT)):     # 1周分
    pose = step(pose, 0.5 - 1.0 * W / 2, 0.5 + 1.0 * W / 2)
    traj.append(pose)
traj = np.array(traj)
print("1周後:", traj[-1], "(始点付近に戻るはず)")

## デッドレコニングの誤差蓄積

車輪速度の指令値だけから位置を推定すると(オドメトリ)、実際の車輪速度とのわずかなズレが積分されて漂流します。これが外界センサによる自己位置推定(`mobile-robots.md` のベイズフィルタ)が必要な理由です。

In [ ]:
# デッドレコニング: 車輪速度にノイズが乗ると推定位置が漂流する
rng = np.random.default_rng(0)
true_pose = np.array([0.0, 0.0, 0.0])
est_pose  = np.array([0.0, 0.0, 0.0])
drift = []
for t in range(2000):
    v_l, v_r = 0.4, 0.5                       # 緩い左カーブ
    noise = rng.normal(0, 0.02, 2)            # 実際の車輪速度のばらつき
    true_pose = step(true_pose, v_l + noise[0], v_r + noise[1])
    est_pose  = step(est_pose,  v_l, v_r)     # 指令値だけから推定
    drift.append(np.linalg.norm(true_pose[:2] - est_pose[:2]))
plt.figure(figsize=(6, 3))
plt.plot(np.arange(len(drift)) * DT, drift)
plt.xlabel("time [s]"); plt.ylabel("position error [m]")
plt.title("dead reckoning drift")
plt.show()
print(f"40s後の誤差: {drift[-1]:.3f} m → 外界センサによる補正(自己位置推定)が必要")

## 課題2: pure pursuit

経路上の**前方注視点**(現在位置から距離 $L_d$ だけ先の点)を選び、そこを通る円弧の曲率

$$\kappa = \frac{2 y_L}{L_d^2}$$

($y_L$ は注視点のロボット座標系での横位置)から $\omega = v\kappa$ を計算します。座標変換の符号を間違えると逆に曲がるので、初期位置から経路に「近づく」ことを必ず確認すること。

In [ ]:
def pure_pursuit(pose, path, lookahead=0.5, v=0.4, w=W):
    """課題2の解答: pure pursuit 追従制御。
    path (N,2) 上の前方注視点を選び、曲率から左右車輪速度を返す"""
    x, y, th = pose
    d = np.linalg.norm(path - pose[:2], axis=1)
    i_near = int(np.argmin(d))
    # 最近傍点から先で、距離が lookahead 以上になる最初の点を注視点にする
    i_target = i_near
    while i_target < len(path) - 1 and d[i_target] < lookahead:
        i_target += 1
    px, py = path[i_target]
    # ---- ここから課題2 ----
    # (1) 注視点 (px, py) をロボット座標系に変換して横ずれ y_L を求める
    # (2) 曲率 kappa = 2 y_L / L_d^2、omega = v * kappa
    #     (L_d はロボットから注視点までの実距離。0除算に注意)
    omega = None
    # ---- ここまで ----
    return v - omega * w / 2, v + omega * w / 2, i_target

# 半径2mの円路を追従する
ts = np.linspace(0, 2 * np.pi, 400)
path = np.stack([2 * np.cos(ts), 2 * np.sin(ts)], axis=1)
pose = np.array([2.2, -0.4, np.pi / 2])       # 経路から少し外れた初期位置
traj, cte = [], []
for t in range(3000):
    v_l, v_r, _ = pure_pursuit(pose, path)
    pose = step(pose, v_l, v_r)
    traj.append(pose.copy())
    cte.append(abs(np.linalg.norm(pose[:2]) - 2.0))   # 円路との横方向誤差
traj = np.array(traj)
print(f"横方向誤差: 序盤1s平均 = {np.mean(cte[:50]):.3f} m, 終盤10s平均 = {np.mean(cte[-500:]):.3f} m")

plt.figure(figsize=(5, 5))
plt.plot(path[:, 0], path[:, 1], "r--", label="path")
plt.plot(traj[:, 0], traj[:, 1], "b", label="robot")
plt.axis("equal"); plt.legend(); plt.title("pure pursuit")
plt.show()

## 発展課題

1. 前方注視距離 $L_d$ を変えて追従誤差と挙動の違いを観察する(下のセル)
2. 経路を四角形(角がある経路)にすると $L_d$ の大小で何が起きるか
3. 車輪速度に上限(例: ±1 m/s)を入れると急カーブでどうなるか

レポートには「結果」と「理由」を結びつけて書くこと。

In [ ]:
# 発展課題1: 前方注視距離 lookahead の影響を調べる
for ld in [0.2, 0.5, 1.5]:
    pose = np.array([2.2, -0.4, np.pi / 2])
    cte = []
    for t in range(3000):
        v_l, v_r, _ = pure_pursuit(pose, path, lookahead=ld)
        pose = step(pose, v_l, v_r)
        cte.append(abs(np.linalg.norm(pose[:2]) - 2.0))
    print(f"lookahead={ld}: 終盤10s平均誤差 = {np.mean(cte[-500:]):.4f} m")
# 小さい→経路に忠実だが振動しやすい / 大きい→滑らかだがコーナーを内側にショートカットする

## まとめ

- 差動二輪は2つの車輪速度だけで平面運動する。ただし真横には動けない(非ホロノミック拘束)
- オドメトリの誤差は蓄積する。POMDPの信念状態(`../reinforcement-learning/pomdp.md`)と同じ構造で外界センサと融合する
- pure pursuit は実装が単純で頑健な経路追従の定番。$L_d$ が実質唯一のチューニングパラメータ
- ナビゲーションをRLで解く定式化は `mobile-robots.md` の最終節を参照